# 03 · How much history does Chronos-2 actually use?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/03_context_length.ipynb)

Companion notebook for *Chronos-2 in Practice (2/5)*.

**What you will learn**

- The paper's hypothesis about long context, and how to turn it into a testable prediction
  about *your* series before running anything.
- Error against context length from 64 to 8,192 steps, at two horizons, with bootstrap
  intervals.
- Why the answer depends on the horizon, and what that implies for the context you should
  actually pay for.

Runtime: ~15 min on a GPU, longer on a CPU (FAST_MODE: ~2 min).

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time

import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.baselines import SeasonalNaive
from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import spain_energy_frame, validate_and_prepare
from c2p.eval import backtest, evaluate, skill_score, summarize
from c2p.experiments import DAILY_SEASON, FREQ, LOAD, WEEKLY_SEASON, weekly_sample_config
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plotting import plot_metric_vs_param, save_figure, set_style
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)  # True: fewer windows and fewer context lengths
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")

## 1. The hypothesis, stated before the experiment

Chronos-2 was pretrained with a context of 2,048 steps and then post-trained to 8,192. The paper
reports that the extension helped "particularly on the GIFT-Eval benchmark, which contains many
high-frequency datasets with long seasonal periods"; the skill score there moved by about a
point.

That is a claim about a *kind* of series, so it makes a prediction about this one. Spanish
hourly load has three cycles: daily (24 steps), weekly (168) and yearly (about 8,766). A context
shorter than 168 cannot see a full week, so it cannot know that Sunday is not Wednesday. A
context of 8,192 steps is just under a year, so it almost sees the yearly cycle but never a
whole one.

The prediction, then: error should fall steeply up to a few multiples of 168 and then flatten,
with little left to gain from 4,096 or 8,192, *unless* the horizon is long enough that the
weekly shape of the forecast matters, in which case more context should keep helping. We test
both horizons rather than assuming.

Two deliberate choices. The sweep varies the model's own `context_length`, which is how many of
the most recent steps Chronos-2 reads; each window's frame is capped at 8,192 steps so that
every setting is fed from the same frame. And latency is recorded but not plotted here: article
5 benchmarks speed properly, and mixing the two questions is how "longer context is slower"
becomes an argument against an accuracy result.

In [ ]:
CONTEXTS = (256, 2048, 8192) if FAST_MODE else (64, 128, 256, 512, 1024, 2048, 4096, 8192)
HORIZONS = (24, WEEKLY_SEASON)  # one day ahead, one week ahead
MAX_CONTEXT = max(CONTEXTS)

raw = spain_energy_frame()[[ID_COL, TIMESTAMP_COL, LOAD]]
series, report = validate_and_prepare(raw, freq=FREQ, target=LOAD)
print(f"Contexts: {CONTEXTS}")
print(f"Horizons: {HORIZONS} hours")
for horizon in HORIZONS:
    config = weekly_sample_config(FAST_MODE, prediction_length=horizon, max_context=MAX_CONTEXT)
    print(f"  horizon {horizon:>3} h -> {config.n_windows} windows, {config.stride} h apart")

## 2. The sweep

For every horizon, Seasonal Naive runs once as the reference and Chronos-2 runs once per context
length. The skill score of each setting is measured against that reference on exactly the same
windows, so the bootstrap intervals are paired.

In [ ]:
pipeline = load_chronos2(device=DEVICE)
baseline = SeasonalNaive(season_length=DAILY_SEASON)


def sweep_horizon(horizon: int) -> pd.DataFrame:
    """Skill of every context length against Seasonal Naive, on one horizon's windows."""
    config = weekly_sample_config(FAST_MODE, prediction_length=horizon, max_context=MAX_CONTEXT)
    reference = evaluate(backtest(baseline, series, config), series, config)
    rows = []
    for context_length in CONTEXTS:
        model = Chronos2Forecaster(
            pipeline, name=f"ctx={context_length}", context_length=context_length
        )
        start = time.perf_counter()
        metrics = evaluate(backtest(model, series, config), series, config)
        seconds = time.perf_counter() - start
        paired = pd.concat([metrics, reference], ignore_index=True)
        skill = skill_score(paired, model=model.name, baseline=baseline.name, metric="WQL")
        means = summarize(metrics).iloc[0]
        rows.append(
            {
                "horizon": horizon,
                "context": context_length,
                "WQL": means["WQL"],
                "MASE": means["MASE"],
                "skill": skill.skill,
                "ci_low": skill.ci_low,
                "ci_high": skill.ci_high,
                "win_rate": skill.win_rate,
                "n_windows": config.n_windows,
                "seconds_total": seconds,
            }
        )
        print(
            f"  horizon {horizon:>3} h, context {context_length:>5}: "
            f"WQL {means['WQL']:.4f}, skill {skill.skill:6.1%} ({seconds:.0f} s)"
        )
    return pd.DataFrame(rows)


start = time.perf_counter()
sweep = pd.concat([sweep_horizon(horizon) for horizon in HORIZONS], ignore_index=True)
print(f"\nSweep finished in {(time.perf_counter() - start) / 60:.1f} min")
print("Saved", save_results(sweep, "03_context_sweep", PATHS).name)

## 3. Error against context

The band is the 95% bootstrap interval of the skill score over windows. Overlapping bands mean
the difference between two context lengths is not resolved by this many windows.

In [ ]:
sweep = load_results("03_context_sweep", PATHS)
labels = {24: "24 h ahead (day-ahead)", WEEKLY_SEASON: f"{WEEKLY_SEASON} h ahead (one week)"}
sweep["Horizon"] = sweep["horizon"].map(labels)

best = sweep.loc[sweep.groupby("horizon")["skill"].idxmax()].set_index("horizon")
gains = {
    horizon: 100
    * (group.set_index("context")["skill"].max() - group.set_index("context")["skill"].min())
    for horizon, group in sweep.groupby("horizon")
}
title = (
    f"Best context: {int(best.loc[24, 'context']):,} steps at 24 h, "
    f"{int(best.loc[WEEKLY_SEASON, 'context']):,} at {WEEKLY_SEASON} h "
    f"({gains[WEEKLY_SEASON]:.1f} points between worst and best)"
)
ax = plot_metric_vs_param(
    sweep,
    x="context",
    y="skill",
    hue="Horizon",
    band=("ci_low", "ci_high"),
    colors={},
    logx=True,
    title=title,
    xlabel="Context length (steps the model reads)",
    ylabel="Skill score vs seasonal naive",
)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
(figure_path,) = save_figure(ax.figure, "03_context_vs_skill", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 4. What the numbers say

The table below is the same data, with the step-by-step gain from doubling the context. The
question an engineer actually has is not "does more context help" but "where does it stop
paying", and that is the column to read.

In [ ]:
sweep = load_results("03_context_sweep", PATHS)
for horizon, group in sweep.groupby("horizon"):
    group = group.sort_values("context")
    print(f"\nHorizon {horizon} h, {int(group['n_windows'].iloc[0])} windows")
    print(f"{'context':>8} {'WQL':>8} {'skill':>8} {'win':>6} {'gain vs previous':>18}")
    previous = None
    for row in group.itertuples():
        gain = "" if previous is None else f"{100 * (row.skill - previous):+.2f} points"
        print(f"{row.context:>8,} {row.WQL:8.4f} {row.skill:8.1%} {row.win_rate:6.0%} {gain:>18}")
        previous = row.skill

## 5. Summary

- The context length is a real knob, not a formality: the spread between the worst and the best
  setting is printed above, in skill points against the same baseline.
- Its value depends on the horizon. A one-day forecast needs enough history to place the day
  within the week; a one-week forecast has more shape to get right, so it keeps using history
  for longer.
- Whatever the sweep says, read the bootstrap intervals before picking a winner: adjacent
  context lengths are often indistinguishable at 52 windows.
- Cost is the other half of the trade and is measured in article 5. `seconds_total` is in the
  saved CSV if you want a preview.

**Next:** [`04_multivariate_coldstart`](04_multivariate_coldstart.ipynb): what to do when a
series has almost no history at all.